# Interpreting and Explaining a QSAR Model

This notebook accompanies the QSAR tutorial chapter: **Interpreting and Explaining a QSAR Model**.

## Prepare data and train a Random Forest model

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import Descriptors

def find_example_dataset():
    candidates = [
        Path("../data/example_qsar_dataset.csv"),
        Path("data/example_qsar_dataset.csv"),
        Path("../../data/example_qsar_dataset.csv"),
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("Could not find example_qsar_dataset.csv.")

def mol_from_smiles(smiles):
    try:
        return Chem.MolFromSmiles(smiles)
    except Exception:
        return None

def calculate_all_rdkit_descriptors(mol):
    return Descriptors.CalcMolDescriptors(mol, missingVal=np.nan)

# Load data
df = pd.read_csv(find_example_dataset())

# Convert SMILES to RDKit molecules and remove invalid molecules
df["Mol"] = df["SMILES"].apply(mol_from_smiles)
df_clean = df.dropna(subset=["Mol"]).copy()

# Calculate all default RDKit descriptors
descriptor_rows = [
    calculate_all_rdkit_descriptors(mol)
    for mol in df_clean["Mol"]
]

X = pd.DataFrame(descriptor_rows, index=df_clean.index)

# Convert to numeric and remove descriptor columns containing missing values
X = X.apply(pd.to_numeric, errors="coerce")
X = X.dropna(axis=1)

# Target values
y = df_clean["Activity"].astype(float)

print("Number of valid molecules:", len(df_clean))
print("Number of RDKit descriptors:", X.shape[1])

X.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np

selector = VarianceThreshold(threshold=1e-8)
X_var = pd.DataFrame(
    selector.fit_transform(X),
    columns=X.columns[selector.get_support()],
    index=X.index
)

top_features = X_var.var().sort_values(ascending=False).head(20).index
X_model = X_var[top_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.25,
    random_state=42
)

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

model.fit(X_train, y_train)

print("Training R2:", r2_score(y_train, model.predict(X_train)))
print("Test R2:", r2_score(y_test, model.predict(X_test)))

## Feature importance

In [ ]:
importance_df = pd.DataFrame({
    "Descriptor": X_train.columns,
    "Importance": model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

display(importance_df.head(10))

In [ ]:
import matplotlib.pyplot as plt

top10 = importance_df.head(10).iloc[::-1]

plt.figure(figsize=(7, 5))
plt.barh(top10["Descriptor"], top10["Importance"])
plt.xlabel("Feature importance")
plt.ylabel("Descriptor")
plt.title("Top 10 Descriptor Importances")
plt.show()

## SHAP values

In [ ]:
try:
    import shap

    explainer = shap.Explainer(model, X_train)
    shap_values = explainer(X_test)

    shap.summary_plot(
        shap_values,
        X_test
    )

except ModuleNotFoundError:
    print("SHAP is not installed. Install it with: pip install shap")

## Local SHAP explanation for one compound

In [ ]:
try:
    import shap

    compound_index = 0

    print("Explaining test compound:")
    print(df_clean.loc[X_test.index[compound_index], ["Name", "SMILES", "Activity"]])

    shap.plots.waterfall(
        shap_values[compound_index]
    )

except NameError:
    print("Run the previous SHAP cell first.")
except ModuleNotFoundError:
    print("SHAP is not installed. Install it with: pip install shap")

## Accumulated Local Effects using a simple custom function

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def ale_1d(model, X, feature, bins=10):
    '''
    Simple 1D Accumulated Local Effects implementation.
    This is for teaching/demo purposes.
    '''
    X = X.copy()
    values = X[feature].values

    # Quantile-based bin edges
    quantiles = np.linspace(0, 1, bins + 1)
    edges = np.unique(np.quantile(values, quantiles))

    effects = []
    centers = []

    for lower, upper in zip(edges[:-1], edges[1:]):
        if lower == upper:
            continue

        mask = (values >= lower) & (values <= upper)

        if mask.sum() == 0:
            continue

        X_lower = X.loc[mask].copy()
        X_upper = X.loc[mask].copy()

        X_lower[feature] = lower
        X_upper[feature] = upper

        pred_upper = model.predict(X_upper)
        pred_lower = model.predict(X_lower)

        local_effect = np.mean(pred_upper - pred_lower)

        effects.append(local_effect)
        centers.append((lower + upper) / 2)

    accumulated = np.cumsum(effects)

    # Center ALE around zero
    accumulated = accumulated - np.mean(accumulated)

    return pd.DataFrame({
        feature: centers,
        "ALE": accumulated
    })

descriptor_name = importance_df.iloc[0]["Descriptor"]
ale_df = ale_1d(model, X_train, descriptor_name, bins=8)

display(ale_df)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(ale_df[descriptor_name], ale_df["ALE"], marker="o")
plt.xlabel(descriptor_name)
plt.ylabel("Accumulated local effect")
plt.title("ALE Plot for " + descriptor_name)
plt.show()

## Interpretation prompt

Look at the ALE plot and decide whether the descriptor has:

- a positive effect,
- a negative effect,
- a nonlinear effect,
- or little effect on the model prediction.